# 🛒 Notebook 03 — AI: Market Basket Analysis & KNN Basket Prediction
### Smart Retail Shelf — IVP + AI Team Project

**Owner role:** AI Lead

This notebook covers the two AI deliverables:

1. **Association Rule Mining** — load the one-hot transaction matrix from
   Notebook 01 (`transactions_encoded.csv`), run `apriori`
   (`min_support=0.01`), extract rules filtered by `lift >= 1.0`, and
   display the **top 10 rules sorted by Lift**.
2. **KNN Basket-Similarity Prediction** — fit
   `sklearn.neighbors.NearestNeighbors` (`metric='hamming'`,
   `n_neighbors=3`) on the same binary matrix. Given a sample "current
   basket state" vector, retrieve the 3 most similar historical baskets —
   useful for "customers who bought like you also bought..." style
   recommendations.

> Runs **out-of-the-box**: if `transactions_encoded.csv` from Notebook 01
> isn't found (e.g. running this notebook standalone), a mock encoded
> transaction matrix is generated automatically so the notebook never
> errors.


## 0. Setup — Install & Import Dependencies

In [ ]:
# ============================================================
# 0.1  Install required libraries (Colab-safe, quiet)
# ============================================================
!pip install -q mlxtend scikit-learn pandas numpy matplotlib


In [ ]:
# ============================================================
# 0.2  Imports
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.neighbors import NearestNeighbors

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

print("Libraries imported successfully ✅")


## 1. Load the One-Hot Transaction Matrix

Expects `transactions_encoded.csv` produced by `preprocessing.ipynb`
(Notebook 01). A fallback mock generator keeps this notebook independently
runnable in Colab if that file isn't present in the current session.


In [ ]:
# ============================================================
# 1.1  Load Notebook 01's output, with a self-contained mock fallback
# ============================================================
CSV_PATH = "transactions_encoded.csv"

def generate_mock_encoded_transactions(n_transactions=40, seed=SEED):
    """
    Fallback: regenerates a plausible one-hot transaction matrix with the
    same co-purchase "affinity" structure as Notebook 01, so Apriori has
    meaningful patterns to discover even in standalone mode.
    """
    catalog = ["Milk", "Bread", "Eggs", "Butter", "Cheese", "Yogurt",
               "Chips", "Soda", "Cereal", "Coffee", "Tea", "Sugar",
               "Rice", "Pasta", "Tomato Sauce", "Chicken", "Frozen Pizza", "Ice Cream"]
    affinity_sets = [
        ["Bread", "Butter", "Milk", "Eggs"],
        ["Pasta", "Tomato Sauce", "Cheese"],
        ["Chips", "Soda", "Ice Cream"],
        ["Coffee", "Tea", "Sugar"],
        ["Chicken", "Rice", "Frozen Pizza"],
    ]
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_transactions):
        basket = set()
        group_idxs = rng.choice(len(affinity_sets), size=rng.integers(1, 3), replace=False)
        for gi in group_idxs:
            group = affinity_sets[gi]
            keep = rng.choice(group, size=rng.integers(2, len(group) + 1), replace=False)
            basket.update(keep.tolist())
        n_extra = rng.integers(0, 3)
        if n_extra > 0:
            basket.update(rng.choice(catalog, size=n_extra, replace=False).tolist())
        row = {item: int(item in basket) for item in catalog}
        rows.append(row)
    return pd.DataFrame(rows)

try:
    encoded_df = pd.read_csv(CSV_PATH)
    # csv round-trips booleans as 0/1 ints already; enforce dtype just in case
    encoded_df = encoded_df.astype(int)
    print(f"Loaded '{CSV_PATH}' from Notebook 01 -> shape {encoded_df.shape}")
except FileNotFoundError:
    encoded_df = generate_mock_encoded_transactions()
    print(f"'{CSV_PATH}' not found -> generated a mock encoded transaction matrix "
          f"-> shape {encoded_df.shape}")

encoded_df.head(10)


## 2. Association Rule Mining (Apriori)
### 2.1 Mine Frequent Itemsets (`min_support = 0.01`)


In [ ]:
# ============================================================
# 2.1  Frequent itemset mining with Apriori
#      min_support=0.01 -> keep itemsets appearing in >=1% of baskets
#      (deliberately low so a small demo dataset still yields itemsets)
# ============================================================
# mlxtend expects boolean dtype for best performance/compatibility
encoded_bool_df = encoded_df.astype(bool)

frequent_itemsets = apriori(
    encoded_bool_df,
    min_support=0.01,
    use_colnames=True,
)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)

print(f"Found {len(frequent_itemsets)} frequent itemset(s).")
frequent_itemsets.head(10)


### 2.2 Extract Association Rules (`lift >= 1.0`)

In [ ]:
# ============================================================
# 2.2  Derive association rules from the frequent itemsets
#      Filter to lift >= 1.0 -> keep only itemsets with NON-negative
#      (i.e. neutral-or-positive) correlation between antecedent & consequent
# ============================================================
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0,
    num_itemsets=len(frequent_itemsets),  # required by newer mlxtend versions
)

# Keep the columns most relevant for a retail "next-item" recommendation
rules_display = rules[[
    "antecedents", "consequents", "support", "confidence", "lift"
]].copy()

# Convert frozenset -> readable comma-separated strings
rules_display["antecedents"] = rules_display["antecedents"].apply(lambda s: ", ".join(sorted(s)))
rules_display["consequents"] = rules_display["consequents"].apply(lambda s: ", ".join(sorted(s)))

print(f"Extracted {len(rules_display)} association rule(s) with lift >= 1.0.")


### 2.3 Top 10 Association Rules Sorted by Lift

In [ ]:
# ============================================================
# 2.3  Display the top 10 rules by Lift (strongest co-purchase signals)
# ============================================================
top10_rules = rules_display.sort_values("lift", ascending=False).head(10).reset_index(drop=True)

# Round for readability
top10_rules[["support", "confidence", "lift"]] = top10_rules[["support", "confidence", "lift"]].round(3)

print("Top 10 Association Rules (sorted by Lift):\n")
top10_rules


In [ ]:
# ============================================================
# 2.4  Quick visualization: Lift of the top rules
# ============================================================
if len(top10_rules) > 0:
    labels = [f"{a} -> {c}" for a, c in zip(top10_rules["antecedents"], top10_rules["consequents"])]
    plt.figure(figsize=(9, 5))
    plt.barh(labels[::-1], top10_rules["lift"][::-1], color="#4285F4")
    plt.xlabel("Lift")
    plt.title("Top 10 Association Rules by Lift — Smart Retail Shelf")
    plt.axvline(1.0, color="gray", linestyle="--", linewidth=1, label="lift = 1.0 (independence)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No rules found at the current thresholds — try lowering min_support.")


## 3. KNN Basket-Similarity Prediction

Frames "basket prediction" as a **similarity search**: given the customer's
*current* (possibly partial) basket, find the `k` most similar historical
transactions using **Hamming distance** (fraction of item-presence flags
that differ) — well-suited to binary/one-hot vectors.


### 3.1 Fit `NearestNeighbors` (metric='hamming', n_neighbors=3)

In [ ]:
# ============================================================
# 3.1  Fit KNN on the binary transaction matrix
# ============================================================
item_columns = encoded_df.columns.tolist()
X = encoded_df[item_columns].values  # shape: (n_transactions, n_items), binary

knn = NearestNeighbors(metric="hamming", n_neighbors=3)
knn.fit(X)

print(f"KNN fitted on {X.shape[0]} historical transactions across {X.shape[1]} unique products.")
print("Metric: hamming | n_neighbors: 3")


### 3.2 Query with a Sample "Current Basket State" Vector

In [ ]:
# ============================================================
# 3.2  Build a sample "current basket" -> encode as the same binary
#      vector space -> find its 3 nearest historical baskets
# ============================================================
# Simulate a customer whose cart currently has these items in progress
sample_basket_items = [item for item in ["Bread", "Butter", "Milk"] if item in item_columns]
if not sample_basket_items:
    # Fallback in case the catalog differs: just pick the first 2 known items
    sample_basket_items = item_columns[:2]

sample_vector = np.zeros((1, len(item_columns)), dtype=int)
for item in sample_basket_items:
    sample_vector[0, item_columns.index(item)] = 1

print("Sample 'current basket state':", sample_basket_items)
print("Encoded vector (first 10 dims):", sample_vector[0][:10])

distances, indices = knn.kneighbors(sample_vector, n_neighbors=3)

print("\nNearest neighbor indices:", indices[0])
print("Hamming distances       :", np.round(distances[0], 4))


### 3.3 Output the 3 Most Similar Historical Baskets

In [ ]:
# ============================================================
# 3.3  Translate neighbor indices back into human-readable item lists
# ============================================================
def vector_to_items(row, columns):
    return [col for col, val in zip(columns, row) if val == 1]

print("=" * 60)
print(f"CURRENT BASKET STATE : {sample_basket_items}")
print("=" * 60)

neighbor_summaries = []
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
    neighbor_items = vector_to_items(X[idx], item_columns)
    # Suggest items present in the neighbor basket but NOT yet in the
    # customer's current basket -> candidate "next item" recommendations
    suggested = [i for i in neighbor_items if i not in sample_basket_items]
    neighbor_summaries.append({
        "rank": rank,
        "transaction_index": int(idx),
        "hamming_distance": round(float(dist), 4),
        "historical_basket": neighbor_items,
        "suggested_additions": suggested,
    })
    print(f"\n#{rank}  (transaction #{idx}, hamming distance = {dist:.4f})")
    print(f"    Historical basket : {neighbor_items}")
    print(f"    Suggested additions for current customer: {suggested}")

neighbors_df = pd.DataFrame(neighbor_summaries)
neighbors_df


---
## ✅ Notebook 03 Summary

| Step | Output |
|---|---|
| Apriori (`min_support=0.01`) | Frequent itemsets table |
| Association rules (`lift >= 1.0`) | Top 10 rules table, sorted by Lift, + bar chart |
| KNN (`hamming`, `k=3`) | 3 most similar historical baskets + suggested next-item additions for a sample current basket |

**Team integration point:** the `suggested_additions` output here is the
natural hook for a "smart shelf" recommendation UI, and the Vision Lead's
YOLOv8 detections (Notebook 02) could feed the *current basket state*
vector in a live deployment instead of the manual sample used here.
